# Project PRISMA-Agent: An Autonomous Multi-Agent Pipeline for Systematic Reviews

**Course:** FDO - M1 Agentic AI & Building AI Agents\
**Technology:** Python, LangChain, LangGraph, OpenRouter APIs, Europe PMC API  
**Level:** PhD Students | **Format:** Jupyter Notebook

## 1. Project Objectives

At the end of this project, the system should be able to:

- Design and implement a modular, stateful multi-agent architecture using **LangGraph**.
- Integrate advanced RAG (Retrieval-Augmented Generation) pipelines for accurate retrieval from scientific corpora (Europe PMC).
- Implement a full **PRISMA 2020**-aligned workflow: search strategy design, literature retrieval, screening (inclusion/exclusion), and data extraction.
- Ensure transparency through detailed logging, citation tracking, and decision rationales (JSON structured outputs).
- Enable human-in-the-loop oversight while maximizing agent autonomy.
- Deliver a general-purpose system that works for any given research topic with minimal configuration.


## 2. Team Division & Individual Contributions

To ensure modularity and parallel development, the 8-agent pipeline is divided into two distinct blocks:

- **Block 1 (Agents 1 to 4) — Data Acquisition & Pre-processing:** Realized by **Ikram Lefhal Lalaoui**.
  *(Focus: Coordinator, Search Strategy, Retrieval, and Deduplication)*
  
- **Block 2 (Agents 5 to 8) — Analysis, Extraction & Generation:** Realized by **Tawfiq Moussif**.
  *(Focus: Screening, Data Extraction, Critic Review, and Final PRISMA Report Synthesis)*



## 1. Environment Setup and LLM Initialization

Before building the multi-agent pipeline, we must configure the working environment. This step installs the required libraries (langgraph, langchain, requests) and securely initializes the Large Language Model (LLM). We use the **OpenRouter API** to connect to a free, high-capability model. The LLM will act as the reasoning engine for all subsequent agents.

In [1]:
!pip install -q langgraph langchain-openai langchain-community requests pandas

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
import os
from getpass import getpass
from typing import TypedDict, List
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
import requests
import json
import pandas as pd

# 1. Clé API OpenRouter (sécurisée)
if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("Entrez votre clé OpenRouter : ")

# 2. Initialisation du LLM (le cerveau des agents)
llm = ChatOpenAI(
    model="openrouter/free",
    temperature=0.1, # Température basse pour des réponses précises
    api_key=os.environ.get("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
)
print("LLM prêt à l'emploi !")

Entrez votre clé OpenRouter : ··········
LLM prêt à l'emploi !


## 2. Shared Memory: The ReviewState

In LangGraph, agents communicate through a shared state. This `ReviewState` is a `TypedDict` that acts as a central memory. It stores the research topic, the generated search queries, and the lists of papers. As the data flows from Agent 2 to Agent 3 and Agent 4, the state is progressively populated. The final fields (screened_papers, extracted_data, final_report) are reserved for Block 2 (Agents 5-8).

In [3]:
class ReviewState(TypedDict):
    topic: str
    search_queries: List[str]
    retrieved_papers: List[dict]
    deduplicated_papers: List[dict]
    screened_papers: List[dict]
    extracted_data: List[dict]
    final_report: str

## 3. Agent 2: Search Strategy Agent

The first functional node of our graph is the Search Strategy Agent. Its role is to transform a raw research topic into valid Boolean queries suitable for academic databases like PubMed or Europe PMC. We use **prompt engineering** and a strict output format to force the LLM to return a clean JSON list of queries, preventing it from generating conversational text.

In [4]:
def search_strategy_agent(state: ReviewState):
    print(f"\n🧠 [Agent 2: Search Strategy] Génération des requêtes pour : '{state['topic']}'")

    prompt = f"""
    You are an expert systematic review librarian.
    Generate exactly 2 optimal academic search queries (Boolean format) for the given topic.
    Output ONLY a valid JSON list of strings. Example: ["query 1", "query 2"]

    Topic: {state['topic']}
    """

    response = llm.invoke(prompt)

    try:
        # Nettoyage pour extraire le JSON même si le LLM ajoute du texte
        content = response.content.replace("```json", "").replace("```", "").strip()
        queries = json.loads(content)
        print(f" Requêtes générées : {queries}")
        return {"search_queries": queries}
    except Exception as e:
        print(f" Erreur de format. On utilise le sujet par défaut.")
        return {"search_queries": [f"({state['topic']})"]}

## 4. Agent 3: Retrieval Agent (Europe PMC API)

Once the Boolean queries are generated, the Retrieval Agent takes over. It connects to the **Europe PMC REST API** to fetch real scientific literature. The agent parses the API's JSON response, extracts the essential metadata (ID, Title, Abstract, Publication Year), and stores them in the state. Robust error handling (`try/except`) is implemented to handle network timeouts or API rate limits gracefully.

In [5]:
def retrieval_agent(state: ReviewState):
    print(f"\n [Agent 3: Retrieval] Recherche des articles sur Europe PMC...")
    queries = state.get("search_queries", [])
    all_papers = []

    for query in queries[:2]: # On limite à 2 requêtes pour ne pas surcharger
        print(f"  -> Recherche : {query}")
        url = f"https://www.ebi.ac.uk/europepmc/webservices/rest/search?query={query}&format=json&pageSize=5"

        try:
            response = requests.get(url, timeout=10)
            response.raise_for_status()
            data = response.json()

            for paper in data.get("resultList", {}).get("result", []):
                all_papers.append({
                    "id": paper.get("id"),
                    "title": paper.get("title", "No Title"),
                    "abstract": paper.get("abstractText", "No abstract available."),
                    "year": paper.get("pubYear", "Unknown")
                })
        except Exception as e:
            print(f"  Erreur API : {e}")

    print(f" [Agent 3] {len(all_papers)} articles bruts récupérés.")
    return {"retrieved_papers": all_papers}

## 5. Agent 4: Deduplication Agent

Scientific databases often return overlapping results for different queries. The Deduplication Agent ensures data quality by cleaning the retrieved dataset. It uses a Python `set` to track unique article IDs and filters out any duplicates. This guarantees that downstream agents (Screening and Extraction) only process unique papers, saving computational resources and preventing bias.

In [6]:
def deduplication_agent(state: ReviewState):
    print(f"\n [Agent 4: Deduplication] Nettoyage des doublons...")
    papers = state.get("retrieved_papers", [])

    unique_papers = []
    seen_ids = set()

    for p in papers:
        if p["id"] not in seen_ids:
            seen_ids.add(p["id"])
            unique_papers.append(p)

    print(f" [Agent 4] {len(unique_papers)} articles uniques conservés (sur {len(papers)} bruts).")
    return {"deduplicated_papers": unique_papers}

## 6. Agent 1: Coordinator (LangGraph Assembly)

This cell represents the Coordinator Agent, built using LangGraph's `StateGraph`. It acts as the orchestrator by defining the workflow logic. We register the previously defined agents as **nodes** and connect them using **edges**. The execution flow is strictly linear: `Search Strategy` → `Retrieval` → `Deduplication` → `END`.

In [7]:
# --- CONSTRUCTION DU GRAPHE (Bloc 1) ---
workflow = StateGraph(ReviewState)

# Ajout des agents
workflow.add_node("search_strategy", search_strategy_agent)
workflow.add_node("retrieval", retrieval_agent)
workflow.add_node("deduplication", deduplication_agent)

# Connexion des étapes (Le flux de travail)
workflow.set_entry_point("search_strategy") # On commence par la stratégie
workflow.add_edge("search_strategy", "retrieval") # Puis on cherche
workflow.add_edge("retrieval", "deduplication") # Puis on nettoie
workflow.add_edge("deduplication", END) # Fin de votre partie

# Compilation du système
app = workflow.compile()
print(" Système Multi-Agents (Bloc 1) compilé avec succès !")

 Système Multi-Agents (Bloc 1) compilé avec succès !


## 7. Pipeline Execution and Data Serialization

Finally, we test the complete Block 1 pipeline. We define an initial state with a sample research topic and invoke the graph. The system runs autonomously through all four agents. At the end, the results are displayed using a Pandas DataFrame, and the deduplicated papers are serialized into a `papers_for_screening.json` file. This JSON file serves as the hand-off point to Block 2 (Agents 5-8).

In [8]:
# Sujet de recherche
initial_state = {
    "topic": "The effect of artificial intelligence on medical diagnosis accuracy",
    "search_queries": [],
    "retrieved_papers": [],
    "deduplicated_papers": [],
    "screened_papers": [],
    "extracted_data": [],
    "final_report": ""
}

print(" Lancement du pipeline de recherche (Agents 1 à 4)...")
final_state = app.invoke(initial_state)

print("\n" + "="*50)
print(" RÉSULTAT FINAL DE VOTRE BLOC")
print("="*50)
print(f"Articles prêts pour le Screening (Agent 5 de votre binôme) : {len(final_state['deduplicated_papers'])}")

# Affichage des 3 premiers titres trouvés
df = pd.DataFrame(final_state['deduplicated_papers'])
if not df.empty:
    print("\nAperçu des articles trouvés sur Europe PMC :")
    print(df[['year', 'title']].head(3).to_string(index=False))
else:
    print("Aucun article trouvé. Essayez de changer de sujet.")

 Lancement du pipeline de recherche (Agents 1 à 4)...

🧠 [Agent 2: Search Strategy] Génération des requêtes pour : 'The effect of artificial intelligence on medical diagnosis accuracy'
 Requêtes générées : ['("artificial intelligence" OR "machine learning" OR "deep learning" OR "neural network*") AND ("medical diagnosis" OR "clinical diagnosis" OR "diagnostic accuracy" OR "diagnostic performance") AND (accuracy OR sensitivity OR specificity OR "diagnostic error")', '(AI OR "computer-aided diagnosis" OR "computer-aided detection" OR "automated diagnosis") AND ("diagnostic accuracy" OR "diagnostic performance" OR "clinical diagnosis") AND (effect OR impact OR improvement OR comparison OR validation)']

 [Agent 3: Retrieval] Recherche des articles sur Europe PMC...
  -> Recherche : ("artificial intelligence" OR "machine learning" OR "deep learning" OR "neural network*") AND ("medical diagnosis" OR "clinical diagnosis" OR "diagnostic accuracy" OR "diagnostic performance") AND (accuracy OR 

## 7. Pipeline Execution and Data Serialization

Finally, we test the complete Block 1 pipeline. We define an initial state with a sample research topic and invoke the graph. The system runs autonomously through all four agents. At the end, the results are displayed using a Pandas DataFrame, and the deduplicated papers are serialized into a `papers_for_screening.json` file. This JSON file serves as the hand-off point to Block 2 (Agents 5-8).

In [9]:
import json

# Sauvegarder les articles dans un fichier
with open("papers_for_screening.json", "w", encoding="utf-8") as f:
    json.dump(final_state['deduplicated_papers'], f, indent=4)

print("Les 9 articles ont été sauvegardés dans 'papers_for_screening.json'.")
print("Vous pouvez télécharger ce fichier depuis la barre latérale gauche et l'envoyer à votre binôme.")

Les 9 articles ont été sauvegardés dans 'papers_for_screening.json'.
Vous pouvez télécharger ce fichier depuis la barre latérale gauche et l'envoyer à votre binôme.


In [10]:
from google.colab import files
import os

# 1. Vérifier que le fichier existe
json_filename = "papers_for_screening.json"

if os.path.exists(json_filename):
    print(f"✅ Fichier trouvé : {json_filename}")
    print("⬇️ Téléchargement du fichier JSON en cours...")
    files.download(json_filename)
    print("✅ Téléchargement terminé ! Vérifiez le dossier 'Téléchargements' de votre ordinateur.")
else:
    print(f"❌ Erreur : Le fichier '{json_filename}' n'existe pas.")
    print("Assurez-vous d'avoir bien exécuté la cellule qui sauvegarde les données (avec le code `json.dump`).")

✅ Fichier trouvé : papers_for_screening.json
⬇️ Téléchargement du fichier JSON en cours...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Téléchargement terminé ! Vérifiez le dossier 'Téléchargements' de votre ordinateur.


# Block 2 — Agents 5–8 (added, Block 1 preserved)

In [11]:
!pip -q install requests langgraph

In [ ]:
"""PRISMA-Agent Block 2: evidence-traceable prototype (Agents 5-8).

Run after Block 1 has exported papers_for_screening.json.
Install: pip install requests langgraph
"""
import csv
import html
import json
import os
import re
import time
import xml.etree.ElementTree as ET
from collections import Counter
from pathlib import Path
from typing import TypedDict, List

import requests
from langgraph.graph import StateGraph, END

TOPIC = "Artificial intelligence for diagnostic accuracy in medical imaging"
SEARCH_QUERY = ('(("artificial intelligence" OR "deep learning" OR "machine learning") '
                'AND (diagnos* OR detect*) AND (sensitivity OR specificity OR accuracy) '
                'AND (imaging OR radiology OR MRI OR CT OR ultrasound OR mammography)) NOT (REVIEW:Y)')
SEARCH_API = "https://www.ebi.ac.uk/europepmc/webservices/rest/search"
OUTPUT = Path('block2_outputs')
OUTPUT.mkdir(parents=True, exist_ok=True)
MAX_NEW_RECORDS = 80               # set to 50 to replicate the previous search
MAX_FULL_TEXT_ATTEMPTS = 25         # prevents excessive network calls
DELAY_BETWEEN_REQUESTS = 0.15
HEADERS = {"User-Agent": "PRISMA-Agent educational-research-prototype/1.0"}

AI_RE = re.compile(r'\b(artificial intelligence|machine learning|deep learning|neural network|cnn|convolutional|radiomics|computer.aided|\bAI\b)\b', re.I)
DIAG_RE = re.compile(r'\b(diagnos\w*|detect\w*|screen\w*|sensitiv\w*|specific\w*|classif\w*|accuracy|auc|roc)\b', re.I)
IMAGE_RE = re.compile(r'\b(imag\w*|radiolog\w*|x.ray|radiograph\w*|ultrasound|sonograph\w*|mammograph\w*|tomograph\w*|\bCT\b|\bMRI\b|\bPET\b|\bCBCT\b|dermoscop\w*|fundus|endoscop\w*)\b', re.I)
REVIEW_RE = re.compile(r'\b(systematic review|meta.analysis|scoping review|umbrella review|rapid review|narrative review|review of reviews)\b', re.I)
PRIMARY_RE = re.compile(r'\b(retrospective|prospective|cohort|case.control|patients? were|participants? were|images? were|dataset|validation cohort|external validation|cross.validation|training set|test set|we evaluated|we developed|we trained)\b', re.I)
METRIC_RE = re.compile(r'\b(sensitivity|specificity|accuracy|roc.auc|auroc|area under the (?:receiver operating characteristic )?curve|\bAUC\b|f1.score|precision)\b', re.I)
METRIC_NUMBER_RE = re.compile(r'\b(sensitivity|specificity|accuracy|AUROC|AUC|F1(?:.score)?|precision)\b\s*(?:of|was|were|=|:|at)?\s*(\d{1,3}(?:\.\d+)?\s*%|0\.\d+|1\.0+)(?!\d)', re.I)


def clean(text):
    text = html.unescape(str(text or ''))
    text = re.sub(r'<[^>]*>', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()


def save_json(filename, data):
    (OUTPUT / filename).write_text(json.dumps(data, indent=2, ensure_ascii=False), encoding='utf-8')


def normalize(p):
    ab = clean(p.get('abstract') or p.get('abstractText') or '')
    if ab.lower() in ('no abstract available.', 'no abstract available', 'none'):
        ab = ''
    publication = str(p.get('firstPublicationDate') or '')
    return {
        'id': str(p.get('id') or p.get('pmid') or p.get('pmcid') or p.get('doi') or ''),
        'title': clean(p.get('title')),
        'abstract': ab,
        'year': str(p.get('year') or publication[:4] or ''),
        'doi': str(p.get('doi') or '').strip(),
        'pmid': str(p.get('pmid') or '').strip(),
        'pmcid': str(p.get('pmcid') or '').strip(),
        'isOpenAccess': str(p.get('isOpenAccess') or '').upper() == 'Y',
        'journalTitle': str(p.get('journalTitle') or ''),
        'authorString': str(p.get('authorString') or ''),
    }


def key_for(p):
    return (('doi:' + p['doi'].lower()) if p['doi'] else
            ('pmid:' + p['pmid']) if p['pmid'] else
            ('title:' + re.sub(r'\W+', '', p['title'].lower())))


def deduplicate(records):
    # Merge all available record identifiers, choosing the record with the richest abstract.
    records = [normalize(p) for p in records]
    groups = []
    index = {}
    for p in records:
        keys = [x for x in [('doi:' + p['doi'].lower() if p['doi'] else ''),
                             ('pmid:' + p['pmid'] if p['pmid'] else ''),
                             ('pmcid:' + p['pmcid'].upper() if p['pmcid'] else ''),
                             ('title:' + re.sub(r'\W+', '', p['title'].lower()))] if x]
        matches = {index[k] for k in keys if k in index}
        if not matches:
            i = len(groups)
            groups.append(p)
        else:
            i = min(matches)
            for old in matches:
                if old != i:
                    other = groups[old]
                    if other:
                        for field in ('doi', 'pmid', 'pmcid'):
                            if not groups[i][field]:
                                groups[i][field] = other[field]
                        if len(other['abstract']) > len(groups[i]['abstract']):
                            groups[i]['abstract'] = other['abstract']
                        groups[old] = None
                        for k, v in list(index.items()):
                            if v == old:
                                index[k] = i
            current = groups[i]
            for field in ('doi', 'pmid', 'pmcid', 'journalTitle', 'authorString'):
                if not current[field]:
                    current[field] = p[field]
            if len(p['abstract']) > len(current['abstract']):
                current['abstract'] = p['abstract']
            current['isOpenAccess'] = current['isOpenAccess'] or p['isOpenAccess']
        for k in keys:
            index[k] = i
        selected = groups[i]
        for k in [x for x in [('doi:' + selected['doi'].lower() if selected['doi'] else ''),
                               ('pmid:' + selected['pmid'] if selected['pmid'] else ''),
                               ('pmcid:' + selected['pmcid'].upper() if selected['pmcid'] else '')] if x]:
            index[k] = i
    return [p for p in groups if p and p['title']]


def fetch_more_papers(max_records=MAX_NEW_RECORDS):
    papers, cursor = [], '*'
    session = requests.Session()
    session.headers.update(HEADERS)
    while len(papers) < max_records:
        resp = session.get(SEARCH_API, params={'query': SEARCH_QUERY, 'format': 'json',
                           'resultType': 'core', 'pageSize': min(100, max_records-len(papers)),
                           'cursorMark': cursor}, timeout=35)
        resp.raise_for_status()
        data = resp.json()
        batch = data.get('resultList', {}).get('result', [])
        if not batch:
            break
        papers.extend(batch)
        nxt = data.get('nextCursorMark')
        if not nxt or nxt == cursor:
            break
        cursor = nxt
        time.sleep(DELAY_BETWEEN_REQUESTS)
    return papers[:max_records]


def fetch_full_text(p, session):
    """Only fetch lawful Europe PMC OA full-text XML when a PMCID is available."""
    pmcid = p.get('pmcid', '')
    if not pmcid:
        return None, 'PMCID not available'
    url = f'https://www.ebi.ac.uk/europepmc/webservices/rest/{pmcid}/fullTextXML'
    try:
        resp = session.get(url, timeout=25)
        if resp.status_code in (404, 403):
            return None, f'Full text XML unavailable (HTTP {resp.status_code})'
        resp.raise_for_status()
        root = ET.fromstring(resp.content)
        body = root.find('.//body')
        if body is None:
            return None, 'XML has no article body'
        sections = []
        for sec in body.findall('.//sec'):
            title = clean(' '.join(sec.find('title').itertext())) if sec.find('title') is not None else 'Body'
            paras = [clean(' '.join(node.itertext())) for node in sec.findall('./p')]
            text = ' '.join(v for v in paras if v)
            if text:
                sections.append({'section': title, 'text': text[:20000]})
        if not sections:
            text = clean(' '.join(body.itertext()))
            if text:
                sections = [{'section': 'Body', 'text': text[:100000]}]
        if not sections:
            return None, 'Article body empty'
        return {'url': url, 'sections': sections[:75]}, None
    except (requests.RequestException, ET.ParseError) as exc:
        return None, f'Failed to parse/download full text: {str(exc)[:120]}'


def snippets(sections, pattern, max_hits=3, width=230):
    hits = []
    for sec in sections:
        text = sec['text']
        for m in pattern.finditer(text):
            start, end = max(0, m.start()-90), min(len(text), m.end()+width-90)
            hits.append({'section': sec['section'], 'excerpt': text[start:end].strip()})
            if len(hits) >= max_hits:
                return hits
    return hits


def load_human_decisions():
    """Optional review file with columns id,decision,reason. Only explicit include grants confirmation."""
    path = Path('human_fulltext_decisions.csv')
    if not path.exists():
        return {}
    with path.open(encoding='utf-8-sig', newline='') as f:
        return {row['id'].strip(): {'decision': row['decision'].strip().lower(),
                'reason': row.get('reason', '').strip()} for row in csv.DictReader(f)
                if row.get('id') and row.get('decision')}


class ReviewState(TypedDict, total=False):
    topic: str
    papers: List[dict]
    full_texts: dict
    full_text_errors: dict
    screening_decisions: List[dict]
    provisional_papers: List[dict]
    included_papers: List[dict]
    pending_papers: List[dict]
    extracted_data: List[dict]
    quality_assessment: List[dict]
    prisma_counts: dict
    final_report: str


def agent5_screening(state: ReviewState):
    print('\n' + '='*68 + '\nAGENT 5 — SCREENING (TITLE/ABSTRACT + AVAILABLE FULL TEXT)\n' + '='*68)
    papers = state['papers']
    decisions, candidates, excluded = [], [], []
    for p in papers:
        text = p['title'] + ' ' + p['abstract']
        # Exclude only unambiguous reviews or clearly irrelevant records.
        if REVIEW_RE.search(p['title']):
            excluded.append((p, 'Review article, not a primary study'))
        elif not (AI_RE.search(text) and DIAG_RE.search(text) and IMAGE_RE.search(text)):
            if not p['abstract'] and (AI_RE.search(p['title']) or DIAG_RE.search(p['title'])):
                candidates.append(p)  # insufficient abstract: do not auto-exclude
            else:
                excluded.append((p, 'Does not clearly address AI diagnostic imaging'))
        else:
            candidates.append(p)
    session = requests.Session()
    session.headers.update(HEADERS)
    texts, errors, provisional, included, pending = {}, {}, [], [], []
    human = load_human_decisions()
    for p, reason in excluded:
        decisions.append({'id': p['id'], 'title': p['title'], 'decision': 'excluded_title_abstract',
                          'reason': reason, 'full_text_url': None})
    for idx, p in enumerate(candidates):
        doc, error = (fetch_full_text(p, session) if idx < MAX_FULL_TEXT_ATTEMPTS
                      else (None, 'Full-text request limit reached'))
        if doc:
            texts[p['id']] = doc
            body = ' '.join(x['text'] for x in doc['sections'])
            # A provisional candidate requires supporting indicators in full text,
            # not simply a promising title. Human confirmation remains separate.
            if AI_RE.search(body) and DIAG_RE.search(body) and IMAGE_RE.search(body) and (PRIMARY_RE.search(body) or METRIC_RE.search(body)):
                status = 'provisional_full_text_eligible'
                reason = 'Full text supports AI diagnostic imaging and primary-study/accuracy features; human verification required'
                provisional.append(p)
            else:
                status = 'pending_human_review'
                reason = 'Full text available, but eligibility cannot be resolved safely by rules'
                pending.append(p)
        else:
            errors[p['id']] = error
            status, reason = 'pending_full_text', error
            pending.append(p)
        human_choice = human.get(p['id'])
        if human_choice and human_choice['decision'] == 'include' and doc:
            status = 'human_confirmed_included'
            reason = 'Manual confirmation: ' + (human_choice['reason'] or 'No rationale supplied')
            included.append(p)
            pending = [x for x in pending if x['id'] != p['id']]
            provisional = [x for x in provisional if x['id'] != p['id']]
        elif human_choice and human_choice['decision'] == 'exclude':
            status = 'human_excluded_full_text'
            reason = 'Manual exclusion: ' + (human_choice['reason'] or 'No rationale supplied')
            pending = [x for x in pending if x['id'] != p['id']]
            provisional = [x for x in provisional if x['id'] != p['id']]
        decisions.append({'id': p['id'], 'title': p['title'], 'decision': status,
                          'reason': reason, 'full_text_url': doc['url'] if doc else None})
        if idx < MAX_FULL_TEXT_ATTEMPTS:
            time.sleep(DELAY_BETWEEN_REQUESTS)
    counts = Counter(x['decision'] for x in decisions)
    prisma = {'records_screened': len(papers),
             'excluded_title_abstract': counts['excluded_title_abstract'],
             'human_excluded_full_text': counts['human_excluded_full_text'],
             'provisional_full_text_eligible': counts['provisional_full_text_eligible'],
             'human_confirmed_included': counts['human_confirmed_included'],
             'pending_full_text_or_review': counts['pending_full_text'] + counts['pending_human_review'],
             'full_texts_retrieved': len(texts)}
    print(f"Screened: {len(papers)} | Title/abstract excluded: {prisma['excluded_title_abstract']}")
    print(f"Full texts retrieved: {len(texts)} | Provisional full-text eligible: {len(provisional)}")
    print(f"Human-confirmed included: {len(included)} | Awaiting review/full text: {prisma['pending_full_text_or_review']}")
    for entry in decisions[:8]:
        print(f"  [{entry['decision']}] {entry['title'][:85]}")
    print('NOTE: Provisional eligibility is NOT final inclusion.')
    return {'screening_decisions': decisions, 'full_texts': texts,
            'full_text_errors': errors, 'provisional_papers': provisional,
            'included_papers': included, 'pending_papers': pending,
            'prisma_counts': prisma}


def agent6_extraction(state: ReviewState):
    print('\n' + '='*68 + '\nAGENT 6 — EVIDENCE / PICO EXTRACTION\n' + '='*68)
    # Extract from provisional full-text candidates, but clearly label as unverified.
    eligible = {p['id']: p for p in state['provisional_papers'] + state['included_papers']}
    records = []
    for id_, p in eligible.items():
        doc = state['full_texts'].get(id_)
        if not doc:
            continue
        sections = doc['sections']
        matches = []
        for sec in sections:
            for match in METRIC_NUMBER_RE.finditer(sec['text']):
                left = max(0, match.start()-85)
                right = min(len(sec['text']), match.end()+130)
                matches.append({'metric': match.group(1), 'reported_value': match.group(2),
                                'section': sec['section'],
                                'source_excerpt': sec['text'][left:right].strip()})
                if len(matches) >= 12:
                    break
            if len(matches) >= 12:
                break
        metric_hits = [{'section': x['section'], 'excerpt': x['source_excerpt']} for x in matches]
        record = {'id': id_, 'title': p['title'], 'doi': p['doi'], 'pmcid': p['pmcid'],
                  'source_url': doc['url'],
                  'verification': ('human_confirmed' if any(x['id'] == id_ for x in state['included_papers'])
                                   else 'provisional_needs_human_check'),
                  'population_evidence': snippets(sections, re.compile(r'\b(patients?|participants?|cohort|dataset|images?)\b', re.I), 2),
                  'index_test_evidence': snippets(sections, AI_RE, 2),
                  'reference_standard_evidence': snippets(sections, re.compile(r'\b(reference standard|ground truth|histopathology|biopsy|radiologist consensus|pathology)\b', re.I), 2),
                  'outcome_evidence': metric_hits[:5],
                  'metric_mentions': matches,
                  'structured_pico_verified': False}
        records.append(record)
    print(f'Full-text evidence records: {len(records)} (human-confirmed: {len(state["included_papers"])})')
    for r in records[:8]:
        print(f"  {r['id']}: {r['title'][:68]} | numeric metric mentions: {len(r['metric_mentions'])} | {r['verification']}")
    if not records:
        print('No usable full-text candidates; extraction is empty, not fabricated.')
    else:
        print('Each record includes source excerpts; metric mentions are NOT validated pooled study results.')
    return {'extracted_data': records}


def agent7_quality(state: ReviewState):
    print('\n' + '='*68 + '\nAGENT 7 — QUADAS-2 QUALITY-APPRAISAL WORKSHEETS\n' + '='*68)
    worksheets = []
    domains = {
        'patient_selection': re.compile(r'\b(consecutive|random sample|retrospective|prospective|inclusion criteria|exclusion criteria)\b', re.I),
        'index_test': re.compile(r'\b(blind\w*|masked|threshold|cut.off|pre.specified)\b', re.I),
        'reference_standard': re.compile(r'\b(reference standard|histopathology|pathology|biopsy|ground truth)\b', re.I),
        'flow_and_timing': re.compile(r'\b(interval|follow.up|excluded from analysis|missing data|all patients)\b', re.I),
    }
    for r in state['extracted_data']:
        sections = state['full_texts'][r['id']]['sections']
        worksheets.append({'id': r['id'], 'title': r['title'], 'source_url': r['source_url'],
                           'tool': 'QUADAS-2', 'assessment_status': 'requires_expert_review',
                           'domains': {name: {'risk_of_bias': 'unclear', 'applicability': 'unclear' if name != 'flow_and_timing' else 'not_applicable',
                                             'supporting_excerpts': snippets(sections, pattern, 2)}
                                       for name, pattern in domains.items()},
                           'caution': 'Excerpts are prompts for appraisal, NOT automated QUADAS-2 judgments.'})
    print(f'QUADAS-2 worksheets created: {len(worksheets)}')
    print('Four domains per study: patient selection; index test; reference standard; flow/timing.')
    print('Risk judgments intentionally UNCLEAR until a qualified reviewer evaluates the methods.')
    for r in worksheets[:5]:
        evidence_domains = sum(bool(d['supporting_excerpts']) for d in r['domains'].values())
        print(f"  {r['id']} — {evidence_domains}/4 domains have supporting text candidates")
    return {'quality_assessment': worksheets}


def agent8_synthesis(state: ReviewState):
    print('\n' + '='*68 + '\nAGENT 8 — GROUNDED NARRATIVE REPORT / PRISMA COUNTS\n' + '='*68)
    c = state['prisma_counts']
    lines = [f'# {state["topic"]}', '', '## PRISMA-Agent exploratory review (not a finalized systematic review)', '',
             '### Screening and full-text retrieval',
             f'- Unique records screened: {c["records_screened"]}',
             f'- Excluded during title/abstract triage: {c["excluded_title_abstract"]}',
             f'- Open full texts retrieved: {c["full_texts_retrieved"]}',
             f'- Provisional full-text-eligible records (NOT confirmed): {c["provisional_full_text_eligible"]}',
             f'- Included after explicit human confirmation: {c["human_confirmed_included"]}',
             f'- Pending full-text retrieval or human decision: {c["pending_full_text_or_review"]}',
             f'- Excluded after human full-text review: {c["human_excluded_full_text"]}', '',
             'These counts are provisional, not a complete PRISMA 2020 flow diagram.', '',
             '### Source-linked exploratory evidence',
             'The following evidence excerpts are from retrieved full texts and require verification.']
    if not state['extracted_data']:
        lines.append('No verifiable full-text evidence was available for extraction.')
    for rec in state['extracted_data']:
        lines += ['', f'#### {rec["title"]}',
                  f'- Identifier: {rec["id"]}; Full text: {rec["source_url"]}',
                  f'- Eligibility: {rec["verification"]}; PICO verification: pending.']
        if rec['metric_mentions']:
            for m in rec['metric_mentions'][:3]:
                quote = m['source_excerpt'].replace('"', "'")[:230]
                lines.append(f'- Text reports {m["metric"]} {m["reported_value"]} (section: {m["section"]}): "{quote}"')
        else:
            lines.append('- No automatically recognized numerical diagnostic-accuracy statement in fetched text.')
    lines += ['', '### Quality / risk of bias',
              f'- QUADAS-2 reviewer worksheets generated: {len(state["quality_assessment"])}.',
              '- No automated low/high risk ratings are asserted: every domain awaits expert review.', '',
              '### Limitations',
              '- Heuristic screening is not independent dual-reviewer screening.',
              '- Full text is only retrieved when openly available from Europe PMC; unavailable PDFs remain pending.',
              '- Numeric metric mentions might concern training, validation, comparison methods or unrelated values.',
              '- No pooling or conclusions about diagnostic effectiveness are justified without verified study-level outcomes.',
              '- A human reviewer must check eligibility, PICO, QUADAS-2 judgments and citations.', '',
              '### Human-in-the-loop confirmation',
              'For clearly eligible full texts, create human_fulltext_decisions.csv containing id,decision,reason',
              '(decision = include or exclude) and rerun. This records explicit human approval; it does not',
              'replace independent dual screening or verified data extraction.']
    report = '\n'.join(lines) + '\n'
    print(f"Narrative draft created ({len(state['extracted_data'])} source-linked evidence records).")
    print(f"Final included: {c['human_confirmed_included']} | Provisional: {c['provisional_full_text_eligible']}")
    print('Report describes limitations rather than inventing findings.')
    return {'final_report': report}


def run_block2():
    source = Path('papers_for_screening.json')
    original = json.loads(source.read_text(encoding='utf-8')) if source.exists() else []
    if isinstance(original, dict):
        original = original.get('papers', original.get('deduplicated_papers', []))
    print('BLOCK 2 INPUT | Original Block 1 records:', len(original))
    try:
        new = fetch_more_papers()
        print('EXPANDED SEARCH | New search records:', len(new))
    except requests.RequestException as exc:
        new = []
        print('EXPANDED SEARCH | Network unavailable:', str(exc)[:180])
    papers = deduplicate(original + new)
    if not papers:
        raise RuntimeError('No papers: upload papers_for_screening.json or enable Europe PMC access.')
    print('EXPANDED SEARCH | Unique papers:', len(papers))
    save_json('expanded_papers.json', papers)

    graph = StateGraph(ReviewState)
    for name, fn in [('agent5', agent5_screening), ('agent6', agent6_extraction),
                     ('agent7', agent7_quality), ('agent8', agent8_synthesis)]:
        graph.add_node(name, fn)
    graph.set_entry_point('agent5')
    graph.add_edge('agent5', 'agent6')
    graph.add_edge('agent6', 'agent7')
    graph.add_edge('agent7', 'agent8')
    graph.add_edge('agent8', END)
    result = graph.compile().invoke({'topic': TOPIC, 'papers': papers})

    for filename, key in [('screening_decisions.json', 'screening_decisions'),
                          ('full_text_errors.json', 'full_text_errors'),
                          ('provisional_papers.json', 'provisional_papers'),
                          ('included_papers.json', 'included_papers'),
                          ('pending_papers.json', 'pending_papers'),
                          ('extracted_data.json', 'extracted_data'),
                          ('quality_assessment.json', 'quality_assessment'),
                          ('prisma_counts.json', 'prisma_counts')]:
        save_json(filename, result[key])
    (OUTPUT / 'final_report.md').write_text(result['final_report'], encoding='utf-8')
    # Review CSV that the user can fill after reading full texts.
    with (OUTPUT / 'human_review_template.csv').open('w', encoding='utf-8', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['id', 'decision', 'reason', 'title', 'full_text_url'])
        for d in result['screening_decisions']:
            if d['decision'] in ('pending_full_text', 'pending_human_review', 'provisional_full_text_eligible'):
                writer.writerow([d['id'], '', '', d['title'], d['full_text_url'] or ''])
    c = result['prisma_counts']
    assert (c['excluded_title_abstract'] + c['human_excluded_full_text'] +
            c['provisional_full_text_eligible'] + c['human_confirmed_included'] +
            c['pending_full_text_or_review'] == c['records_screened']), c
    print('\n' + '='*68 + '\nFINAL CONSISTENCY CHECK — PASSED\n' + '='*68)
    print(json.dumps(c, indent=2))
    print('Saved outputs to:', OUTPUT.resolve())
    print('Human review template:', OUTPUT / 'human_review_template.csv')
    return result


if __name__ == '__main__':
    run_block2()


BLOCK 2 INPUT | Original Block 1 records: 10
EXPANDED SEARCH | New search records: 80
EXPANDED SEARCH | Unique papers: 90

AGENT 5 — SCREENING (TITLE/ABSTRACT + AVAILABLE FULL TEXT)
Screened: 90 | Title/abstract excluded: 37
Full texts retrieved: 18 | Provisional full-text eligible: 18
Human-confirmed included: 0 | Awaiting review/full text: 35
  [excluded_title_abstract] Artificial Intelligence-Enhanced Electrocardiography for the Detection of Hypertrophi
  [excluded_title_abstract] A comparison of diagnostic accuracy between CNN and transformer AI models for periodo
  [excluded_title_abstract] Artificial Intelligence for Automated Analysis and Interpretation of Intravascular Ul
  [excluded_title_abstract] The Role of Artificial Intelligence Combined With Digital Cholangioscopy for Indeterm
  [excluded_title_abstract] Artificial Intelligence in Gastrointestinal Endoscopy: Current Advances, Clinical Int
  [excluded_title_abstract] Artificial Intelligence Versus Human Interpretation in 